# Cerebrum Weather Assistant Example

This notebook demonstrates how to use the Cerebrum framework as a library to create a weather assistant that provides clothing recommendations based on weather conditions.

## Setup and Imports

First, let's import the necessary modules from the Cerebrum framework.

In [1]:
import sys
from pathlib import Path
from datetime import datetime

# Add the project root to the Python path
project_root = str(Path().resolve().parents[0])
sys.path.append(project_root)

from src.server.persistence.persistence_model_factory import PersistenceModelFactory
from src.server.models.tasks_manager import TasksManager
from src.server.models.task import Task
from src.server.models.message import Message
from src.server.controllers.curried_proxy import CurriedProxy
from src.server.tools.weather_tool import WeatherTool
from src.server.controllers.task_runner import TaskRunner

/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/crewai_tools/tools/scrapegraph_scrape_tool/scrapegraph_scrape_tool.py:34: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprec


🔧 Initializing default tools...

📦 Registering internal tools...
Registering instance internal_file_tools
  Registered method: internal_file_tools.read_documentation
  Registered method: internal_file_tools.save_implementation
  Registered method: internal_file_tools.save_tests
✅ Registered InternalFileTools with 4 methods

☁️ Registering weather tools...
Registering instance weather
  Registered method: weather.get_forecast
  Registered method: weather.get_weather
✅ Registered WeatherTool with 3 methods

🐍 Registering Python execution tool...
Registering instance PyExec
  Registered method: PyExec.run_code
  Registered method: PyExec.run_script
✅ Registered PyExec with 3 methods
Registering instance internal_credentials_tools
  Registered method: internal_credentials_tools.load_credentials
  Registered method: internal_credentials_tools.validate_credentials
✅ Registered InternalCredentialsTools with 3 methods
Registering instance internal_testing_tools
  Registered method: internal_t

2025-03-29 23:56:18.820 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-29 23:56:18.820 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2025-03-29 23:56:18.820 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-29 23:56:18.820 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


🆕 Creating new persistence model
🆕 Created InMemoryPersistenceModel: id=6043024272
✅ Stored new persistence model in session state: id=6043024272


/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/PyPDF2/__init__.py:21: DeprecationWarning: PyPDF2 is deprecated. Please move to the pypdf library instead.
  warnings.warn(


## Initialize Persistence Model

Create an in-memory persistence model to store our tasks and tools.

In [ ]:
# Initialize persistence model
persistence_model = PersistenceModelFactory.persistence_model("inmemory")
print("✅ Initialized in-memory persistence model")

## Create and Register Weather Tools

Set up the weather tools and register them with a curried proxy.

In [ ]:
# Create weather tool
weather_tool = WeatherTool()

# Create a curried proxy for the tools
curried_proxy = CurriedProxy()
curried_proxy.add_tool("weather.get_weather", weather_tool.get_weather)
curried_proxy.add_tool("weather.get_forecast", weather_tool.get_forecast)

# Save the curried proxy
curried_proxy = persistence_model.curried_proxies.create(curried_proxy)
curried_proxy_id_str = str(curried_proxy.id)
print(f"✅ Created curried proxy with ID: {curried_proxy_id_str}")

## Create Weather Task Manager

Create a task manager for weather-related tasks.

In [ ]:
# Create TasksManager for weather
weather_manager = TasksManager(
    name="WeatherManager",
    model="gpt-4o-mini",
    description="Manages tasks for weather reporting and clothing recommendations.",
    messages=[
        Message(
            role="system",
            content="You are a helpful assistant."
        )
    ]
)
weather_manager = persistence_model.tasksmanagers.create(weather_manager)
print(f"✅ Created Weather Manager with ID: {weather_manager.id}")

## Create Clothing Recommendation Task

Create a task that provides clothing recommendations based on weather conditions.

In [ ]:
# Create task for clothing recommendations
clothing_task = Task(
    name="WhatToWear",
    description="Provides clothing recommendations based on weather conditions",
    tasksmanager_id=str(weather_manager.id),
    curried_proxy_id=curried_proxy_id_str,
    use_context=True,
    inherit_manager_context=True,
    messages=[
        Message(
            role="system",
            content="""You are a clothing recommendation assistant specialized in weather-appropriate attire. Your role is to provide helpful, practical clothing suggestions based on weather conditions.

When making recommendations, consider:

1. Temperature ranges:
   - Hot (>85°F/30°C): Lightweight, breathable fabrics
   - Warm (70-85°F/21-30°C): Light layers, short sleeves
   - Mild (55-70°F/13-21°C): Light jackets, long sleeves
   - Cool (40-55°F/4-13°C): Sweaters, light coats
   - Cold (25-40°F/-4-4°C): Heavy coats, layers, hats
   - Very cold (<25°F/-4°C): Thermal layers, heavy insulation

2. Precipitation:
   - Rain: Waterproof jackets, umbrellas, water-resistant footwear
   - Snow: Waterproof boots, insulated coats, gloves, hats
   - Sleet/Ice: Non-slip footwear, waterproof outer layers

3. Wind conditions:
   - Windy: Wind-resistant layers, secured accessories
   - Gusty: Avoid loose clothing, use windbreakers

4. Humidity levels:
   - High humidity: Moisture-wicking fabrics, breathable materials
   - Low humidity: Hydrating fabrics, skin protection

5. UV exposure:
   - Sunny: Hats, sunglasses, UV-protective clothing
   - Overcast: Still consider UV protection

6. Activity level:
   - Active: Moisture-wicking, flexible clothing
   - Casual: Comfortable, appropriate layers
   - Formal: Weather-appropriate professional attire

Always ask for specific weather details if not provided. Tailor recommendations to the user's location, planned activities, and personal preferences. Provide practical, specific suggestions rather than general advice."""
        )
    ]
)
clothing_task = persistence_model.tasks.create(clothing_task)
print(f"✅ Created Clothing Recommendation Task with ID: {clothing_task.id}")

## Create Task Runner

Initialize the task runner to execute our tasks.

In [ ]:
# Create task runner
task_runner = TaskRunner(persistence_model)
await task_runner.initialize()
print("✅ Created Task Runner")

## Function to Get Weather and Clothing Recommendations

Create a function that takes a location and returns weather information and clothing recommendations.

In [7]:
async def get_weather_and_clothing_recommendations(location: str):
    """Get weather information and clothing recommendations for a location."""
    # Create a new chat session
    session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # First message to get weather information
    weather_query = f"What is the weather in {location}?"
    
    # Create a Message object for the weather query
    weather_message = Message(role="user", content=weather_query)
    
    # Run the task with the weather query
    weather_response_data = await task_runner.run_task(
        tasksmanager_id=str(weather_manager.id),
        task_id=str(clothing_task.id),
        running_extra_params={
            "messages": [weather_message],
            "session_id": session_id
        }
    )
    
    # Extract the response from the result
    weather_response = weather_response_data.get("response", "")
    
    # Second message to get clothing recommendations
    clothing_query = f"What should I wear for this weather in {location}?"
    
    # Create a Message object for the clothing query
    clothing_message = Message(role="user", content=clothing_query)
    
    # Run the task with the clothing query
    clothing_response_data = await task_runner.run_task(
        tasksmanager_id=str(weather_manager.id),
        task_id=str(clothing_task.id),
        running_extra_params={
            "messages": [clothing_message],
            "session_id": session_id
        }
    )
    
    # Extract the response from the result
    clothing_response = clothing_response_data.get("response", "")
    
    return {
        "location": location,
        "weather_info": weather_response,
        "clothing_recommendations": clothing_response
    }

## Try It Out

Let's test our weather assistant with different locations.

In [ ]:
import asyncio

# Test with New York
result_ny = await get_weather_and_clothing_recommendations("New York")

print(f"\n🌆 {result_ny['location']} Weather Information:")
print(f"{'=' * 50}")
print(result_ny['weather_info'])
print(f"\n👕 Clothing Recommendations:")
print(f"{'=' * 50}")
print(result_ny['clothing_recommendations'])

In [ ]:
# Test with London
result_london = await get_weather_and_clothing_recommendations("London")

print(f"\n🌆 {result_london['location']} Weather Information:")
print(f"{'=' * 50}")
print(result_london['weather_info'])
print(f"\n👕 Clothing Recommendations:")
print(f"{'=' * 50}")
print(result_london['clothing_recommendations'])

## Create a Function for Interactive Weather Assistant

Let's create a function that allows users to interactively query the weather assistant.

In [10]:
# from IPython.display import clear_output

# async def interactive_weather_assistant():
#     """Interactive weather assistant that provides weather information and clothing recommendations."""
#     print("🌤️ Welcome to the Cerebrum Weather Assistant! 👕")
#     print("This assistant provides weather information and clothing recommendations.")
#     print("Type 'exit' to quit.\n")
    
#     session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
#     conversation_history = []
    
#     while True:
#         user_input = input("You: ")
#         if user_input.lower() == 'exit':
#             print("\nThank you for using the Cerebrum Weather Assistant! Goodbye!")
#             break
        
#         # Create a Message object for the user input
#         user_message = Message(role="user", content=user_input)
        
#         # Add user message to history
#         conversation_history.append({"role": "user", "content": user_input})
        
#         # Get response from the assistant
#         response_data = await task_runner.run_task(
#             tasksmanager_id=str(weather_manager.id),
#             task_id=str(clothing_task.id),
#             running_extra_params={
#                 "messages": [user_message],
#                 "session_id": session_id
#             }
#         )
        
#         # Extract the response from the result
#         response = response_data.get("response", "")
        
#         # Add assistant response to history
#         conversation_history.append({"role": "assistant", "content": response})
        
#         # Display the response
#         print(f"\nAssistant: {response}\n")
        
#     return conversation_history

In [ ]:
# Run the interactive assistant
# conversation = await interactive_weather_assistant()

## Conclusion

This notebook demonstrates how to use the Cerebrum framework as a library to create a weather assistant that provides clothing recommendations based on weather conditions. We've shown how to:

1. **Create a weather assistant from scratch**:
   - Initialize a persistence model
   - Create and register weather tools
   - Create a task manager and tasks
   - Use a task runner to execute tasks

2. **Interact with the assistant in different ways**:
   - Get weather information and clothing recommendations for specific locations
   - Use an interactive chat interface for natural conversations

This example showcases the flexibility of the Cerebrum framework and how it can be integrated into Python notebooks for interactive applications. The framework provides a powerful way to create AI assistants with specialized capabilities, such as providing weather information and clothing recommendations.

In a full Cerebrum application, this functionality would be available through the Streamlit UI, where users can interact with various specialized states and tasks through a unified interface.